# Docs Ingestion API — interactive client

This notebook calls the backend API defined in [api.py](api.py) using `requests`.

Start the server first, in a terminal, from the repo root:

```bash
uvicorn src.genai.api:app --host 0.0.0.0 --port 8000
```

(or however it's already running — e.g. via `python -m src.genai.application.launcher`,
which puts the backend on `$CB_BACKEND_PORT`, default `8000`.)

Then run the cells below top to bottom.

In [ ]:
import os

import requests

# .strip() guards against a stray trailing space breaking urllib3's host:port parser
# (e.g. "http://127.0.0.1:8000 " raises requests.exceptions.InvalidURL on every call).
BASE_URL = f"http://127.0.0.1:{os.environ.get('CB_BACKEND_PORT', '8000')}".strip()


def pretty(resp):
    print(resp.status_code)
    try:
        from pprint import pprint
        pprint(resp.json())
    except ValueError:
        print(resp.text)

## Health check

In [ ]:
resp = requests.get(f"{BASE_URL}/health")
pretty(resp)

## Generate — single file, CSV output

Processes one `.docx`/`.xlsx`/`.pdf` file and writes/appends to a CSV. Uses one of the
sample files in [data/](data/).

In [ ]:
payload = {
    "file_path": "./data/bank_faq.xlsx",
    "output_path": "questions.csv",
    "mode": "append",
}
resp = requests.post(f"{BASE_URL}/generate", json=payload)
pretty(resp)

## Generate — folder scan, overwrite CSV

Recursively scans a folder for `.docx`/`.xlsx`/`.pdf` files and overwrites the output CSV
instead of appending.

In [ ]:
payload = {
    "folder_path": "./data",
    "output_path": "questions.csv",
    "mode": "overwrite",
    "questions_per_chunk": 5,
}
resp = requests.post(f"{BASE_URL}/generate", json=payload)
pretty(resp)

## Generate — push to Elasticsearch

Same pipeline, but rows are embedded (local `sentence-transformers` model) and bulk-indexed into
Elasticsearch instead of written to CSV. `output_path` is not needed for this sink.
Connection/index settings come from the `[elasticsearch]` section of `config.toml`.

In [ ]:
payload = {
    "folder_path": "./data",
    "output_mode": "elasticsearch",
}
resp = requests.post(f"{BASE_URL}/generate", json=payload)
pretty(resp)

## Error handling examples

- `422` — request body failed validation (e.g. neither/both of `folder_path`/`file_path` given).
- `400` — application-level error (bad path, unsupported file type, missing `output_path` for the `csv` sink, etc.).

In [ ]:
# 422: neither folder_path nor file_path given
resp = requests.post(f"{BASE_URL}/generate", json={"output_path": "questions.csv"})
pretty(resp)

In [ ]:
# 400: folder does not exist
resp = requests.post(
    f"{BASE_URL}/generate",
    json={"folder_path": "./does-not-exist", "output_path": "questions.csv"},
)
pretty(resp)

## Inspect the resulting CSV

Only useful after running one of the `csv`-sink cells above.

In [ ]:
import pandas as pd

df = pd.read_csv("questions.csv")
df.head()